In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# from pathlib import Path
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt

# # =========================================================
# # 설정: Erangel 폴더(맵별 로그 분리된 CSV) 경로
# # =========================================================
# BASE = Path(r"C:\msys64\home\for\10th\00_Project\04_final\02_parquet_file\03_mapwise_logs_kakao+steam_20260211_20260219")
# MAP_DIR = BASE / "Erangel"

# landing_path  = MAP_DIR / "LogParachuteLanding.csv"
# pos_path      = MAP_DIR / "LogPlayerPosition.csv"
# gs_path       = MAP_DIR / "LogGameStatePeriodic.csv"

# POS_SAMPLE_FRAC = 1.0  # 0.2 등으로 낮추면 빠른 EDA 가능

# # =========================================================
# # 1) 필요한 최소 컬럼만 로드
# # =========================================================
# landing_cols = [
#     "matchId", "_D", "character_accountId",
#     "character_location_x", "character_location_y", "character_location_z",
# ]
# pos_cols = [
#     "matchId", "_D", "elapsedTime", "character_accountId", "character_teamId",
#     "character_location_x", "character_location_y", "character_location_z",
#     "character_isInVehicle", "character_isInBlueZone",
# ]
# gs_cols = [
#     "matchId", "_D", "gameState_elapsedTime",
#     "gameState_safetyZonePosition_x", "gameState_safetyZonePosition_y", "gameState_safetyZoneRadius",
# ]

# landing = pd.read_csv(landing_path, usecols=lambda c: c in landing_cols)
# pos     = pd.read_csv(pos_path,     usecols=lambda c: c in pos_cols)
# gs      = pd.read_csv(gs_path,      usecols=lambda c: c in gs_cols)

# if POS_SAMPLE_FRAC < 1.0:
#     pos = pos.sample(frac=POS_SAMPLE_FRAC, random_state=42)

# # =========================================================
# # 2) 타입/결측 정리 (merge_asof 안전화)
# # =========================================================
# landing["_D"] = pd.to_datetime(landing["_D"], errors="coerce", utc=True)
# pos["_D"]     = pd.to_datetime(pos["_D"],     errors="coerce", utc=True)
# gs["_D"]      = pd.to_datetime(gs["_D"],      errors="coerce", utc=True)

# landing = landing.dropna(subset=["matchId","character_accountId","_D"]).copy()
# pos     = pos.dropna(subset=["matchId","character_accountId","_D"]).copy()
# gs      = gs.dropna(subset=["matchId","_D"]).copy()

# # matchId 혼합 타입 방지
# landing["matchId"] = landing["matchId"].astype(str)
# pos["matchId"]     = pos["matchId"].astype(str)
# gs["matchId"]      = gs["matchId"].astype(str)

# # boolean 정리(문자/0/1 섞여도 mean 계산 가능하게)
# def to_bool(s: pd.Series) -> pd.Series:
#     if str(s.dtype) in ("bool", "boolean"):
#         return s.astype("boolean")
#     return (
#         s.astype(str).str.lower()
#          .map({"true": True, "false": False, "1": True, "0": False, "nan": pd.NA, "none": pd.NA})
#          .astype("boolean")
#     )

# pos["character_isInVehicle"] = to_bool(pos["character_isInVehicle"])
# pos["character_isInBlueZone"] = to_bool(pos["character_isInBlueZone"])

# # =========================================================
# # 3) (A) 매치×유저 피처 생성
# # =========================================================

# # 3-1) Landing → drop table (matchId, accountId 1행)
# landing_sorted = landing.sort_values(["matchId","character_accountId","_D"], kind="mergesort").reset_index(drop=True)
# drop = (
#     landing_sorted
#     .groupby(["matchId","character_accountId"], as_index=False)
#     .first()
#     .rename(columns={
#         "character_location_x": "drop_x",
#         "character_location_y": "drop_y",
#         "character_location_z": "drop_z",
#         "_D": "drop_time",
#     })
# )

# # 3-2) Position ⨝ GameState (merge_asof)
# pos_sorted = pos.sort_values(["matchId","_D"], kind="mergesort").reset_index(drop=True)
# gs_sorted  = gs.sort_values(["matchId","_D"], kind="mergesort").reset_index(drop=True)

# try:
#     # 정상 경로(빠름): by="matchId"
#     pos_gs = pd.merge_asof(
#         pos_sorted,
#         gs_sorted,
#         on="_D",
#         by="matchId",
#         direction="backward",
#         allow_exact_matches=True
#     )
# except ValueError:
#     # fallback(확실): matchId별로 쪼개서 merge_asof
#     print("merge_asof(by=matchId) failed -> fallback to per-match merge_asof")
#     parts = []
#     gs_groups = {mid: g.drop(columns=["matchId"]) for mid, g in gs_sorted.groupby("matchId", sort=False)}
#     # ▲ 핵심 수정: 오른쪽(gs)에서 matchId를 제거해서 matchId_x/y가 생기지 않게 함

#     for mid, gpos in pos_sorted.groupby("matchId", sort=False):
#         ggs = gs_groups.get(mid)
#         if ggs is None or ggs.empty:
#             continue
#         merged = pd.merge_asof(
#             gpos, ggs,
#             on="_D",
#             direction="backward",
#             allow_exact_matches=True
#         )
#         # merged에는 gpos의 matchId가 그대로 남아있음
#         parts.append(merged)

#     pos_gs = pd.concat(parts, ignore_index=True)

# # 혹시라도(다른 이유로) matchId가 suffix로 변했을 때 복구
# if "matchId" not in pos_gs.columns:
#     if "matchId_x" in pos_gs.columns:
#         pos_gs["matchId"] = pos_gs["matchId_x"]
#     elif "matchId_y" in pos_gs.columns:
#         pos_gs["matchId"] = pos_gs["matchId_y"]
#     else:
#         raise RuntimeError("merge 결과에 matchId가 없습니다. 컬럼명을 확인하세요.")

# # 서클 정보 없는 행 제거
# pos_gs = pos_gs.dropna(subset=[
#     "gameState_safetyZonePosition_x",
#     "gameState_safetyZonePosition_y",
#     "gameState_safetyZoneRadius"
# ]).copy()

# # 3-3) safezone 정규화 거리(dist_norm)
# dx = pos_gs["character_location_x"] - pos_gs["gameState_safetyZonePosition_x"]
# dy = pos_gs["character_location_y"] - pos_gs["gameState_safetyZonePosition_y"]
# dist = np.sqrt(dx*dx + dy*dy)

# radius = pos_gs["gameState_safetyZoneRadius"].replace(0, np.nan)
# pos_gs["safe_dist_norm"] = dist / radius

# # 3-4) 매치×유저 집계
# tmp = pos_gs.copy()
# tmp["is_edge"] = tmp["safe_dist_norm"] > 0.8
# tmp["is_center"] = tmp["safe_dist_norm"] < 0.3

# match_user = (
#     tmp.groupby(["matchId","character_accountId"], as_index=False)
#        .agg(
#            pos_samples=("safe_dist_norm","size"),
#            safe_norm_mean=("safe_dist_norm","mean"),
#            safe_norm_std=("safe_dist_norm","std"),
#            edge_ratio=("is_edge","mean"),
#            center_ratio=("is_center","mean"),
#            z_std=("character_location_z","std"),
#            vehicle_ratio=("character_isInVehicle","mean"),
#            bluezone_ratio=("character_isInBlueZone","mean"),
#            elapsed_min=("elapsedTime","min"),
#            elapsed_max=("elapsedTime","max"),
#        )
# )

# # drop 피처 merge(landing 없는 유저는 NaN)
# match_user = match_user.merge(
#     drop[["matchId","character_accountId","drop_x","drop_y","drop_z","drop_time"]],
#     on=["matchId","character_accountId"],
#     how="left"
# )

# # =========================================================
# # 4) (B) 유저 프로필 생성 (accountId 단위)
# # =========================================================
# user_profile = (
#     match_user.groupby("character_accountId", as_index=False)
#               .agg(
#                   matches=("matchId","nunique"),
#                   safe_norm_mean=("safe_norm_mean","mean"),
#                   safe_norm_std=("safe_norm_mean","std"),
#                   edge_ratio=("edge_ratio","mean"),
#                   center_ratio=("center_ratio","mean"),
#                   vehicle_ratio=("vehicle_ratio","mean"),
#                   bluezone_ratio=("bluezone_ratio","mean"),
#                   z_std=("z_std","mean"),
#                   drop_x_mean=("drop_x","mean"),
#                   drop_y_mean=("drop_y","mean"),
#                   drop_x_std=("drop_x","std"),
#                   drop_y_std=("drop_y","std"),
#               )
# )

# # =========================================================
# # 5) EDA 시각화
# # =========================================================
# plt.figure()
# plt.scatter(match_user["safe_norm_mean"], match_user["vehicle_ratio"], s=5, alpha = 0.4)
# plt.xlabel("safe_dist_norm mean (match-user)")
# plt.ylabel("vehicle_ratio (match-user)")
# plt.title("Positioning vs Vehicle Dependency (Match-User)")
# plt.show()

# plt.figure()
# plt.scatter(user_profile["edge_ratio"], user_profile["center_ratio"], s=8, alpha = 0.4)
# plt.xlabel("edge_ratio (user)")
# plt.ylabel("center_ratio (user)")
# plt.title("Edge vs Center Tendency (User Profile)")
# plt.show()

# plt.figure()
# plt.scatter(user_profile["vehicle_ratio"], user_profile["bluezone_ratio"], s=8, alpha = 0.4)
# plt.xlabel("vehicle_ratio (user)")
# plt.ylabel("bluezone_ratio (user)")
# plt.title("Vehicle Dependency vs Bluezone (User Profile)")
# plt.show()

# plt.figure()
# plt.hist(user_profile["matches"], bins=30)
# plt.xlabel("matches per user")
# plt.ylabel("count of users")
# plt.title("User Activity Distribution")
# plt.show()

# print("match_user shape:", match_user.shape)
# print("user_profile shape:", user_profile.shape)
# print("\nmatch_user head:")
# print(match_user.head(3))
# print("\nuser_profile head:")
# print(user_profile.head(3))